In [ ]:
import pandas as pd
import numpy as np
import yaml

In [ ]:
# Loading data
with open ('config.yaml', 'r') as file_adress:
    config = yaml.safe_load(file_adress)

# Csv file path
csvFile = config['resultcsv']

In [ ]:
df = pd.read_csv(csvFile)
df = df.dropna(axis=1, how='all')
df = df.dropna(thresh=2).reset_index(drop=True)
df.head(30)

In [ ]:
# Create new 'Min' and 'Max' columns filled with NaN values
df['Min'] = np.nan
df['Max'] = np.nan

# Iterate over each row in the DataFrame
for index, row in df.iterrows():
    value = row['Streeftraject']
    
    # Check if the value is NaN
    if pd.isna(value):
        continue  # Skip this iteration if value is NaN
    
    # Check if the value has a range separated by ' - '
    if ' - ' in value: 
        min, max = value.split(' - ')

        # Remove the comma and space from the numbers
        min = min.replace(',', '')
        min = min.replace(' ', '')
        max = max.replace(',', '')
        max = max.replace(' ', '')

        # Cast the string to a number
        min_val = float(min)
        max_val = float(max)

        df['Min'][index] = min_val
        df['Max'][index] = max_val 
        
    # Check if the value starts with '< '
    elif value.startswith('< '):
        min_val = float('-inf')

        # Remove the unneccessary part from the numbers
        value = value.replace(' ', '')
        value = value.replace('<', '')
        value = value.replace(',', '')

        # Cast the string to a number
        max_val = float(value) 

        df['Min'][index] = min_val
        df['Max'][index] = max_val
    
    # Check if the value starts with '> '
    elif value.startswith('> '): 

        # Remove the unneccessary part from the numbers
        value = value.replace(' ', '')
        value = value.replace('>', '')
        value = value.replace(',', '')

        # Cast the string to a number
        min_val = float(value)

        max_val = float('inf')
        df['Min'][index] = min_val
        df['Max'][index] = max_val

# Remove the original 'Streeftraject' column
df.drop('Streeftraject', axis=1, inplace=True)
df

In [ ]:
# Remove '<' and '>' characters from the 'Resu ltaat' column
df['Resu ltaat'] = df['Resu ltaat'].str.replace('<', '')
df['Resu ltaat'] = df['Resu ltaat'].str.replace('>', '')
df['Resu ltaat'] = df['Resu ltaat'].str.replace(',', '.')
df['Resu ltaat'] = df['Resu ltaat'].str.replace(' ', '')

# Cast the 'Resu ltaat' column to numeric data type
df['Resu ltaat'] = pd.to_numeric(df['Resu ltaat'])

In [ ]:
# Create a new 'Check' column filled with NaN values
df['Check'] = np.nan

# Iterate over each row in the DataFrame
for index, row in df.iterrows():
    result = row['Resu ltaat']
    min_val = row['Min']
    max_val = row['Max']
    
    if pd.isna(result) or pd.isna(min_val) or pd.isna(max_val):
        df['Check'][index] = 'Unsure'
    elif result >= min_val and result <= max_val:
        df['Check'][index] = 'OK'
    else:
        df['Check'][index] = 'Not in range'

df